In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder \
    .appName("Lakehouse - Serving Layer") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262,io.delta:delta-spark_2.12:3.1.0,mysql:mysql-connector-java:8.0.33") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

fact_df = spark.read.format("delta").load("s3a://lakehouse/warehouse/fact_delivery_task_gold")

n = fact_df.count()
print(f"Fact: {n} dòng")
if n == 0:
    raise ValueError("Fact rỗng - chạy lại Silver và Gold trước")

# Cột giờ nhận đơn lấy từ Silver (Fact không có accept_ts)
silver = spark.read.format("delta").load("s3a://lakehouse/warehouse/delivery_silver") \
              .select("order_id", F.hour("accept_ts").alias("accept_hour"))
fact_h = fact_df.join(silver, "order_id", "left")

# 1. Hiệu suất theo shipper
agg_courier = fact_df.groupBy("courier_id", "region_id").agg(
    F.count("order_id").alias("total_orders"),
    F.round(F.avg("distance_km"), 2).alias("avg_distance_km"),
    F.round(F.avg("delivery_minutes"), 2).alias("avg_delivery_minutes"),
    F.round(F.expr("percentile_approx(delivery_minutes, 0.5)"), 2).alias("median_delivery_minutes"),
    F.round(F.max("delivery_minutes"), 2).alias("max_delivery_minutes")
).orderBy(F.desc("total_orders"))

# 2. Theo vùng và ngày (cho biểu đồ xu hướng)
agg_region_daily = fact_df.groupBy("region_id", "order_date").agg(
    F.count("order_id").alias("total_orders"),
    F.countDistinct("courier_id").alias("active_couriers"),
    F.round(F.avg("distance_km"), 2).alias("avg_distance_km"),
    F.round(F.avg("delivery_minutes"), 2).alias("avg_delivery_minutes"),
    F.round(F.expr("percentile_approx(delivery_minutes, 0.5)"), 2).alias("median_delivery_minutes")
).orderBy("region_id", "order_date")

# 3. Theo giờ nhận đơn (cho biểu đồ khung giờ cao điểm)
agg_hourly = fact_h.groupBy("accept_hour").agg(
    F.count("order_id").alias("total_orders"),
    F.round(F.avg("delivery_minutes"), 2).alias("avg_delivery_minutes"),
    F.round(F.expr("percentile_approx(delivery_minutes, 0.5)"), 2).alias("median_delivery_minutes")
).orderBy("accept_hour")

print("=== 5 SHIPPER CHẠY NHIỀU ĐƠN NHẤT ===")
agg_courier.show(5, truncate=False)

jdbc = {
    "url": "jdbc:mysql://mysql:3306/delivery_source",
    "driver": "com.mysql.cj.jdbc.Driver",
    "user": "root",
    "password": "root",
    "truncate": "true",
}

for name, df in [("agg_courier_performance", agg_courier),
                 ("agg_region_daily", agg_region_daily),
                 ("agg_hourly", agg_hourly)]:
    df.write.format("jdbc").options(**jdbc).option("dbtable", name).mode("overwrite").save()
    print(f"Đã đẩy bảng {name} sang MySQL ({df.count()} dòng)")